# Tutorial 4: Cross-Slice Evaluation

This notebook demonstrates CauST's **multi-slice** mode and the **Leave-One-Donor-Out (LODO)** validation protocol — the key benchmark for measuring cross-slice generalization.

## Why Cross-Slice?

A model that works on Slice 1 but fails on Slice 2 (from a different donor) is not reliable. CauST's invariance scoring specifically addresses this by finding genes that are *consistently causal* across all slices and donors.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import pandas as pd

from caust import CauST
from caust.data.loader import load_and_preprocess
from caust.evaluate.metrics import evaluate_single_slice, compute_ari
from caust.causal.invariance import lodo_splits

## 1. Create Multi-Donor Synthetic Data

We simulate the DLPFC-like setup: 3 donors, 2 slices each, 3 spatial domains.
Each donor has slightly different expression profiles (biological variability).

In [ ]:
def make_slice(n_obs=200, n_vars=150, n_domains=3, donor_shift=0.0, seed=42):
    """Create a single tissue slice with donor-specific shifts."""
    rng = np.random.default_rng(seed)
    obs_per = n_obs // n_domains

    X_parts, coord_parts = [], []
    for i in range(n_domains):
        base = rng.poisson(lam=5 + donor_shift, size=(obs_per, n_vars)).astype(np.float32)
        # Causal genes (consistent across donors)
        causal_start = i * 10
        base[:, causal_start:causal_start + 10] += 10 * (i + 1)
        # Confounding genes (donor-specific — NOT causal for domains)
        conf_start = 100 + i * 5
        base[:, conf_start:conf_start + 5] += donor_shift * 3
        X_parts.append(base)
        angle = 2 * np.pi * i / n_domains
        coord_parts.append(rng.normal([12 * np.cos(angle), 12 * np.sin(angle)], 2, (obs_per, 2)))

    X = np.vstack(X_parts)
    adata = ad.AnnData(X=csr_matrix(X))
    adata.obs_names = [f'spot_{i}' for i in range(X.shape[0])]
    adata.var_names = [f'gene_{j}' for j in range(n_vars)]
    adata.obsm['spatial'] = np.vstack(coord_parts)
    adata.obs['ground_truth'] = np.repeat(np.arange(n_domains), obs_per).astype(str)
    return adata

# 3 donors × 2 slices each = 6 slices
slices = {}
donor_map = {}
seed_counter = 0
for donor_idx, donor_name in enumerate(['Donor1', 'Donor2', 'Donor3']):
    donor_shift = donor_idx * 2.0  # each donor has different baseline expression
    for s in range(2):
        sid = f'{donor_name}_s{s+1}'
        slices[sid] = make_slice(donor_shift=donor_shift, seed=seed_counter)
        donor_map[sid] = donor_name
        seed_counter += 10

print(f'Created {len(slices)} slices from {len(set(donor_map.values()))} donors:')
for sid, adata in slices.items():
    print(f'  {sid} (donor={donor_map[sid]}): {adata.n_obs} spots × {adata.n_vars} genes')

## 2. Preprocess All Slices

In [ ]:
slices_pp = {}
for sid, adata in slices.items():
    slices_pp[sid] = load_and_preprocess(adata, n_top_genes=100, min_genes=5, min_cells=1)
    print(f'  {sid}: {slices_pp[sid].n_obs} spots × {slices_pp[sid].n_vars} genes')

## 3. Run CauST Multi-Slice Mode

This is the **recommended** CauST usage. It:
1. Trains a model per slice
2. Computes per-slice causal scores
3. Computes IRM-style invariance scores across all slices
4. Combines causal + invariance → final gene ranking

In [ ]:
model = CauST(
    n_causal_genes=40, n_clusters=3, epochs=60,
    scoring_method='gradient', alpha=0.5,
    filter_mode='filter_and_reweight', verbose=True,
)

results = model.fit_multi_slice(slices_pp, donor_map=donor_map)

print(f'\nResults for {len(results)} slices.')

## 4. Evaluate Per-Slice Performance

In [ ]:
rows = []
for sid, adata_out in results.items():
    labels_pred = adata_out.obs['caust_domain'].astype(int).values
    latent_Z = adata_out.obsm['caust_latent']
    labels_true = slices_pp[sid].obs.loc[adata_out.obs_names, 'ground_truth'].values
    m = evaluate_single_slice(labels_pred, latent_Z, labels_true)
    m['slice'] = sid
    m['donor'] = donor_map[sid]
    rows.append(m)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 5. Leave-One-Donor-Out (LODO) Validation

The gold-standard cross-generalization test:
- Train CauST on all slices from 2 donors
- Test on slices from the held-out donor
- Rotate which donor is held out

In [ ]:
slice_ids = list(slices_pp.keys())
splits = lodo_splits(slice_ids, donor_map)

lodo_rows = []
for train_sids, test_sids in splits:
    test_donor = donor_map[test_sids[0]]
    print(f'\n--- LODO: hold out {test_donor} ---')
    print(f'    Train: {train_sids}')
    print(f'    Test:  {test_sids}')

    # Train on train slices
    train_slices = {sid: slices_pp[sid].copy() for sid in train_sids}
    lodo_model = CauST(
        n_causal_genes=40, n_clusters=3, epochs=60,
        scoring_method='gradient', alpha=0.5,
        filter_mode='filter_and_reweight', verbose=False,
    )
    lodo_model.fit_multi_slice(train_slices)

    # Test on held-out donor's slices
    for sid in test_sids:
        adata_test = lodo_model.transform(slices_pp[sid].copy())
        labels_pred = adata_test.obs['caust_domain'].astype(int).values
        latent_Z = adata_test.obsm['caust_latent']
        labels_true = slices_pp[sid].obs.loc[adata_test.obs_names, 'ground_truth'].values
        m = evaluate_single_slice(labels_pred, latent_Z, labels_true)
        m['test_slice'] = sid
        m['held_out_donor'] = test_donor
        lodo_rows.append(m)
        print(f'    {sid}: ARI={m.get("ari", "N/A"):.4f}')

lodo_df = pd.DataFrame(lodo_rows)
print('\n=== LODO Results ===')
print(lodo_df.to_string(index=False))

## 6. Inspect Invariant Causal Genes

Genes ranked high by CauST should be the **truly causal** ones (gene_0–gene_29) that are consistent across donors, NOT the confounding genes (gene_100+) that vary by donor.

In [ ]:
top = model.get_top_causal_genes(n=20)
print('Top 20 genes after invariance scoring:')
for gene, score in top:
    marker = ''
    idx = int(gene.split('_')[1])
    if idx < 30:
        marker = '  ← CAUSAL (domain marker)'
    elif idx >= 100:
        marker = '  ← confounding (donor-specific)'
    print(f'  {gene}: {score:.4f}{marker}')

## 7. Visualize Cross-Slice Heatmap

In [ ]:
import os
os.makedirs('output', exist_ok=True)

if hasattr(model, 'per_slice_scores'):
    from caust.visualize.plots import plot_invariance_heatmap
    plot_invariance_heatmap(
        model.per_slice_scores,
        top_k=30,
        title='Gene Causal Scores Across 6 Slices',
        out_path='output/cross_slice_heatmap.png',
    )
    print('Heatmap saved to output/cross_slice_heatmap.png')
else:
    print('No per_slice_scores found — run fit_multi_slice first.')

## 8. Visualize LODO ARI Comparison

In [ ]:
if 'ari' in lodo_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    donors = lodo_df['held_out_donor'].unique()
    x = np.arange(len(donors))
    means = [lodo_df[lodo_df['held_out_donor'] == d]['ari'].mean() for d in donors]
    bars = ax.bar(x, means, color='steelblue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(donors)
    ax.set_ylabel('ARI (held-out donor)')
    ax.set_title('LODO Cross-Donor Generalization')
    ax.set_ylim(0, 1.05)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, m + 0.02, f'{m:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

## Summary

| Protocol | What it tests | Why it matters |
|----------|--------------|----------------|
| Same-slice ARI | Quality on training data | Basic sanity check |
| **LODO ARI** | Generalization to unseen donors | **CauST's main value proposition** |
| Invariance heatmap | Visual confirmation of gene stability | Biological interpretability |

CauST's invariance scoring specifically penalises donor-specific confounders and promotes universally causal genes, directly improving LODO performance.